# Building and Refining a Café ReAct Agent

A ReAct agent lets a model request information, receives observations from Python tools, and uses the growing interaction to choose a next action or answer. The orchestrator coordinates that process; it does not prove that the answer is correct. This notebook extends Section 5.2 from one tool interaction to a bounded investigation.

You will reuse the same three CSV functions, describe their inputs, predict the required lookups, run the loop and check its report. Then you may revise one instruction and repeat from a fresh interaction. Run all cells from the top in a fresh kernel. All three CSVs are included. Offline mode follows a fixed, labelled teaching sequence and makes no API requests. Optional live mode uses your private package-root .env and allows at most six requests per investigation; refinement is separately opt-in.

The setup below repeats Section 5.2 unchanged so no other notebook must be open. Leave USE_OPENAI false for browser Python. Live mode needs the local companion requirements and a standard Python kernel.

In [ ]:
import pandas as pd

sales = pd.read_csv("sales.csv")
events = pd.read_csv("campus_events.csv")
promotions = pd.read_csv("promotions.csv")


In [ ]:
def get_sales(date, product_id):
    # Match both the date and the product.
    rows = sales[
        (sales["date"] == date)
        & (sales["product_id"] == product_id)
    ]

    columns = ["date", "product_name", "quantity_sold"]
    return rows[columns].to_dict("records")


In [ ]:
def get_events(date):
    # Retrieve events recorded on the requested date.
    rows = events[events["date"] == date]

    columns = ["date", "event_name", "expected_attendance"]
    return rows[columns].to_dict("records")


In [ ]:
def get_promotions(date, product_id):
    # Exclude promotions for other products.
    rows = promotions[
        (promotions["date"] == date)
        & (promotions["product_id"] == product_id)
    ]

    columns = ["date", "product_id", "discount_percent"]
    return rows[columns].to_dict("records")


In [ ]:
events_tool = {
    "type": "function",
    "name": "get_events",
    "description": (
        "Retrieve recorded campus events for a date. "
        "Attendance figures are expected attendance, "
        "not actual café visits."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "date": {
                "type": "string",
                "description": "Date in YYYY-MM-DD format."
            }
        },
        "required": ["date"],
        "additionalProperties": False
    }
}


## Configure supplied or live mode

This is the same package-root .env setup used in 5.2. The switch is false by default. The next section provides offline infrastructure separately from the loop you are learning.

In [ ]:
import json
import os
from pathlib import Path
from types import SimpleNamespace

# Offline is the default. Set True only when ready for two real API requests.
USE_OPENAI = False
client = None
model = "supplied-example"

if USE_OPENAI:
    import sys
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("Use offline mode in browser Python; live calls need a local Python kernel.")
    from dotenv import load_dotenv
    from openai import OpenAI

    # Section 2.3 setup extended only to locate the downloaded package root.
    config_folder = Path.cwd()
    for folder in [config_folder, *config_folder.parents]:
        if (folder / "COMPANION_VERSION").is_file():
            config_folder = folder
            break
    load_dotenv(config_folder / ".env")
    model = os.environ["OPENAI_MODEL"]
    client = OpenAI(timeout=45.0, max_retries=0)
    print("Live mode configured. Running the next request cell uses OpenAI.")
else:
    print("Offline mode: supplied messages, real Python CSV lookups, no API requests.")


In [ ]:
# Offline infrastructure only: supplied requests are not model decisions.
# This stand-in exposes the same response fields used by the book loop.
def make_supplied_client():
    state = {"step": 0}
    actions = [
        ("get_sales", {"date": "2026-02-10", "product_id": "P01"}),
        ("get_sales", {"date": "2026-02-11", "product_id": "P01"}),
        ("get_events", {"date": "2026-02-11"}),
        ("get_promotions", {"date": "2026-02-11", "product_id": "P01"})
    ]

    def create(**request):
        index = state["step"]
        state["step"] += 1
        if index < len(actions):
            name, arguments = actions[index]
            call = SimpleNamespace(
                type="function_call", name=name,
                arguments=json.dumps(arguments), call_id=f"supplied-{index}"
            )
            return SimpleNamespace(output=[call], output_text="")
        # This is author-written teaching text, not a live model response.
        report = (
            "SUPPLIED REPORT: coffee sales increased by 71 units. A campus gathering "
            "and a coffee promotion were recorded on 11 February. These records "
            "do not establish the cause of the increase."
        )
        if "state the sales quantity" in request["instructions"]:
            report = (
                "SUPPLIED REVISED REPORT: 10 February: 35 coffees; 11 February: 106. "
                "Change: 106 - 35 = 71 coffees. A campus gathering (expected attendance "
                "200) and a 20% coffee discount were recorded. This does not prove causation."
            )
        return SimpleNamespace(output=[], output_text=report)

    return SimpleNamespace(responses=SimpleNamespace(create=create), close=lambda: None)


## Give the Agent an Investigation

We will reuse this question:

> How did coffee sales change between 10 and 11 February 2026, and what recorded events or promotions might be relevant?

The agent can request information through:

- `get_sales(date, product_id)`
- `get_events(date)`
- `get_promotions(date, product_id)`

These are the same functions introduced in Section 5.2. Their implementations remain unchanged.

The model will choose its requests. Python will execute the functions, return their results and ask the model what to do next.

This follows the ReAct approach introduced by Yao et al. [@yao2023react]: actions obtain information that can guide subsequent decisions.

## Describe the Remaining Tools

Section 5.2 provided the event-tool definition. We also need definitions for sales and promotions.

Both functions require a date and product identifier, so we can reuse one parameter description:

In [ ]:
product_parameters = {
    "type": "object",
    "properties": {
        "date": {
            "type": "string",
            "description": "Date in YYYY-MM-DD format."
        },
        "product_id": {
            "type": "string",
            "description": "Product identifier; Coffee is P01."
        }
    },
    "required": ["date", "product_id"],
    "additionalProperties": False
}


Now describe what each function retrieves:

In [ ]:
sales_tool = {
    "type": "function",
    "name": "get_sales",
    "description": (
        "Retrieve recorded sales quantity for a product and date. "
        "An empty result means the sales record is unavailable."
    ),
    "parameters": product_parameters
}

promotions_tool = {
    "type": "function",
    "name": "get_promotions",
    "description": (
        "Retrieve recorded promotions for a product and date. "
        "An empty result means no matching promotion was recorded."
    ),
    "parameters": product_parameters
}

tools = [sales_tool, events_tool, promotions_tool]


The list makes all three tool definitions available to the model. It does not execute any function.

## Set the Agent’s Instructions

The instructions should establish the investigation’s boundaries without prescribing every lookup.

In [ ]:
system_prompt = """
You are a café data analyst.

Use the available tools to investigate the user's question.
Use only information supported by the returned records.
Coffee has product identifier P01.

Compare sales on the requested dates and inspect relevant
event and promotion records before preparing your answer.

If a required record is unavailable, explain what is missing.
Distinguish observed changes from possible explanations.
Keep the final report concise.
"""

question = """
How did coffee sales change between 2026-02-10 and 2026-02-11,
and what recorded events or promotions might be relevant?
"""


Unlike Section 5.2, we will not force a particular tool. The model can select among the available tools and provide their arguments.

The notebook includes the same `.env` configuration and client setup used previously. Create a fresh client before starting; the final cell in Section 5.2 closed its client.

## Understand the Agent Loop

The program repeats a small sequence:

[Table: Steps managed by the café agent loop]

| Stage | What happens |
|---|---|
| Request | Send the question and accumulated interaction to the model. |
| Inspect | Check whether the response requests a tool or contains an answer. |
| Execute | Run the requested Python function. |
| Return | Add its result to the interaction. |
| Continue or stop | Request the next response, or stop after the final answer. |

The code coordinating these steps is commonly called an **orchestrator**. In our example, it is a short Python loop.

The model proposes the next operation; the orchestrator executes it. We can inspect requests and results without treating them as a direct view of the model’s internal reasoning.

## Python Experiment: Run the Investigation

**Predict:** Before running the program, write down:

- Which records do you expect the model to request?
- Will it need one sales record or two?
- What should it do if a lookup returns no matching record?
- What would count as an unsupported explanation?

### Start a New Interaction

Keep the user’s question in a list. As the investigation proceeds, we will add model responses and tool results.

In [ ]:
max_steps = 6
# Each investigation creates its own message list inside run_investigation().

Each step allows one model request. The limit prevents the program from continuing indefinitely.

### Request, Execute and Return

The following code extends Section 5.2’s single interaction. The new part is repetition: after returning an observation, we request another response.

**Notebook extension:** the book shows the loop directly. Here the same loop body is placed inside `run_investigation()` so baseline and revised runs reuse identical code. Its local `messages` list is reset on every call. No automatic refinement is performed.

In [ ]:
def run_investigation(client, model, instructions, question, max_steps=6):
    system_prompt = instructions
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        response = client.responses.create(
            model=model,
            instructions=system_prompt,
            input=messages,
            tools=tools,
            parallel_tool_calls=False,
            max_output_tokens=500,
            store=False
        )
    
        # Retain the complete response for the next request.
        messages += response.output
    
        # Select function requests from the response.
        tool_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]
    
        # No tool request: display the model's answer and stop.
        if not tool_calls:
            if response.output_text.strip():
                print("\nFINAL ANSWER")
                print(response.output_text)
            else:
                print("No answer returned. The investigation is unfinished.")
            break
    
        # This example allows one tool request per model turn.
        tool_call = tool_calls[0]
        arguments = json.loads(tool_call.arguments)
    
        print(f"\nStep {step + 1}")
        print("Requested tool:", tool_call.name)
        print("Arguments:", arguments)
    
        # Execute only the functions explicitly available here.
        if tool_call.name == "get_sales":
            observation = get_sales(
                arguments["date"],
                arguments["product_id"]
            )
    
        elif tool_call.name == "get_events":
            observation = get_events(arguments["date"])
    
        elif tool_call.name == "get_promotions":
            observation = get_promotions(
                arguments["date"],
                arguments["product_id"]
            )
    
        else:
            observation = {"error": "Unknown tool."}
    
        print("Observation:", observation)
    
        # Return the result with its corresponding request identifier.
        messages.append({
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(observation)
        })
    
    else:
        print("\nStep limit reached. The investigation is unfinished.")
    return messages


In [ ]:
if not USE_OPENAI:
    print("SUPPLIED SEQUENCE — no API calls; CSV functions execute normally.")
    client = make_supplied_client()

try:
    baseline_messages = run_investigation(client, model, system_prompt, question, max_steps)
finally:
    client.close()


### Follow the Program Step by Step

**First, request the next response.** The model receives the instructions, question, available tools and previous interaction.

**Second, inspect the response.** A tool request identifies the function and its arguments. Otherwise, the program displays the answer and stops.

**Third, execute the function.** The `if` and `elif` branches connect the requested name to one of our three Python functions.

**Fourth, return the observation.** The result is added with the matching `call_id`.

**Finally, continue.** The next iteration gives the model the accumulated information. It can request another lookup or answer the question.

`parallel_tool_calls=False` keeps the demonstration to one function request per turn. This makes the sequence easier to follow.

## Examine the Investigation

**Run:** Execute the notebook’s live experiment only when ready to make model requests. A run can make up to six requests. Stop and inspect the results before repeating it.

The offline route uses a clearly labelled supplied sequence of requests. Python executes the same CSV lookups, but the sequence does not represent fresh model decisions.

**Observe and Explain:**

- Did the agent retrieve sales for both dates?
- Which event and promotion records did it request?
- Did it repeat an unnecessary lookup?
- Did the final answer agree with the returned records?
- Did it distinguish a possible explanation from an established cause?

The sequence may differ between live runs. Judge whether the required evidence was obtained, rather than expecting one fixed order.

A response without a tool request makes this simple program stop. That does not automatically establish that the investigation was complete or the answer was correct. If the API returns an incomplete response or an empty answer, inspect it before treating the run as successful.

## Check the Sales Calculation Independently

Use Python to verify the quantities and difference in the final report:

In [ ]:
previous = get_sales("2026-02-10", "P01")
current = get_sales("2026-02-11", "P01")

if previous and current:
    expected_change = (
        current[0]["quantity_sold"]
        - previous[0]["quantity_sold"]
    )
    print("Checked change in coffees sold:", expected_change)
else:
    print("Cannot calculate the change: a record is unavailable.")


Also compare the reported event and promotion details with the tool results.

An event’s expected attendance is not actual café attendance. A promotion or event recorded alongside a sales increase does not establish why the increase occurred.

## Refine One Instruction

Refinement should address something observed in the response.

For example, the answer might report a change without clearly identifying the quantities used. We can add:

In [ ]:
additional_instruction = """
In the final report, state the sales quantity for each date
and show the subtraction used to calculate the change.
"""

revised_system_prompt = (
    system_prompt + "\n" + additional_instruction
)


Make this revision only if it addresses a real weakness. If the first answer already meets the requirement, explain that and identify another specific issue—or explain why no revision is needed.

For a fair comparison:

1. Keep the question, datasets, tools and model unchanged.
2. Create a fresh client and reset `messages` to the original question.
3. Run the same loop using `revised_system_prompt`.
4. Compare the tool requests, final report and checked calculation.

A fresh interaction prevents the second run from inheriting the first run’s answers. Each live comparison can incur additional model requests.





## Apply What You Have Learned

Choose another pair of dates. Predict which records will be needed, then inspect the agent’s requests and answer.

Explain one case where the available records support a finding and one conclusion they cannot establish. If working offline, label any report you write yourself as reader-written. The supplied sequence remains fixed to its documented dates.

## Check Your Understanding

1. What does the Python orchestrator do that the model does not?
2. Why are tool results retained in `messages`?
3. What happens when the model returns no tool request?
4. Why does reaching the step limit mean the investigation may be unfinished?
5. Why should a refinement comparison start with a fresh interaction?

**Key takeaway:** A ReAct agent combines model-selected requests with operations executed by the application. The loop returns each result so that the model can choose its next step. Clear instructions help guide the investigation, but the final report must still be checked against the records.

## Optional manual refinement comparison

Read the baseline report first. The supplied example deliberately omits the two source quantities; its revision adds them. This demonstrates a specific improvement, not evidence that longer prompts generally improve a model. If a live response already meets the requirement, explain that and do not invent a weakness. Choose whether to run the revision below. It uses a fresh client and a new message list; in live mode it allows up to six additional requests.

In [ ]:
RUN_REFINEMENT = False

if RUN_REFINEMENT:
    if USE_OPENAI:
        client = OpenAI(timeout=45.0, max_retries=0)
    else:
        client = make_supplied_client()
    try:
        revised_messages = run_investigation(
            client, model, revised_system_prompt, question, max_steps
        )
    finally:
        client.close()
else:
    print("Refinement not run. Inspect the baseline, then choose whether to enable it.")


## Save your findings

Write your prediction, the actual sequence of requested tools, the checked difference and one limitation. If you run the revision, explain what improved and what did not. Offline requests and reports are supplied examples. For a different-date offline investigation, call the functions directly and label your own report reader-written; the supplied client is intentionally fixed to 10–11 February. There is nothing to submit.